# Experiment 7 — Left carotid stent / revascularization surrogate

## Goal

This experiment simulates **left ICA stenosis and post-stent revascularization**.

The question is:

> Does restoring the left ICA lumen recover left MCA and left ACA perfusion, and how do collateral flows change after treatment?

## Cases

| Case | Left ICA area | Left ICA stiffness | Interpretation |
|---|---:|---:|---|
| `healthy` | 100% | baseline | no lesion |
| `pre_stent_85pct_area_stenosis` | 15% | baseline | severe stenosis |
| `post_stent_80pct_area_restored` | 80% | 5× baseline | stented, mostly restored lumen, stiffer wall |
| `post_stent_full_area_restored` | 100% | 5× baseline | idealized lumen recovery with stiffer stented segment |

The stiffness multiplier is a simple surrogate: real stents alter local compliance and create a short treated segment. Here we apply it to the whole `L-ICA_I` segment for readability.

In [ ]:
from __future__ import annotations

import copy
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import hemo1d as hd

MMHG_TO_DYN_CM2 = 1333.22


def mmhg(value: float) -> float:
    return value * MMHG_TO_DYN_CM2


def read_config(path: Path) -> dict:
    with Path(path).open() as f:
        return json.load(f)


def write_config(config: dict, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f:
        json.dump(config, f, indent=2)
    return path


def area_key(vessel: dict) -> str:
    for key in ('initial_area', 'area0', 'A0'):
        if key in vessel:
            return key
    raise KeyError('Vessel has no initial_area/area0/A0 field.')


def get_area(config: dict, vessel_id: str) -> float:
    vessel = config['vessels'][vessel_id]
    return float(vessel[area_key(vessel)])


def set_area(config: dict, vessel_id: str, area: float) -> None:
    vessel = config['vessels'][vessel_id]
    vessel[area_key(vessel)] = float(area)


def scale_area(config: dict, vessel_id: str, fraction: float) -> None:
    base_area = get_area(config, vessel_id)
    set_area(config, vessel_id, base_area * fraction)


def beta_key(vessel: dict) -> str:
    for key in ('beta_coeff', 'beta'):
        if key in vessel:
            return key
    raise KeyError('Vessel has no beta_coeff/beta field.')


def scale_beta(config: dict, vessel_id: str, factor: float) -> None:
    vessel = config['vessels'][vessel_id]
    key = beta_key(vessel)
    vessel[key] = float(vessel[key]) * factor


def set_all_vessel_pressures(config: dict, *, p0_mmhg: float = 80.0, p_ext_mmhg: float = 0.0) -> None:
    """Set a consistent resting reference pressure for every vessel."""
    config.setdefault('defaults', {})
    config['defaults']['p0'] = mmhg(p0_mmhg)
    config['defaults']['p_ext'] = mmhg(p_ext_mmhg)

    for vessel in config['vessels'].values():
        vessel['p0'] = mmhg(p0_mmhg)
        vessel['p_ext'] = mmhg(p_ext_mmhg)


def set_capillary_venous_pressure(config: dict, p_ven_mmhg: float) -> None:
    for bed in config.get('capillary_beds', {}).values():
        bed['P_ven'] = mmhg(p_ven_mmhg)


def set_capillary_initial_pressure(config: dict, p0_mmhg: float) -> None:
    for bed in config.get('capillary_beds', {}).values():
        bed['P0'] = mmhg(p0_mmhg)


def make_pressure_inlet(mean_mmhg: float, amp1_mmhg: float, amp2_mmhg: float, *, phase_s: float = 0.0):
    """Use the source-code waveform creator, not a notebook-local redefinition."""
    return hd.create_arterial_pressure_inflow(
        mean_mmhg=mean_mmhg,
        amp1_mmhg=amp1_mmhg,
        amp2_mmhg=amp2_mmhg,
        heart_rate=1.2,
        ramp_time=0.20,
        phase_s=phase_s,
    )


def install_pressure_inlets(model) -> None:
    model.set_inlet(
        vessel_id='L-ICA_I',
        side='right',
        kind='pressure',
        function=make_pressure_inlet(85.0, 14.0, 4.0, phase_s=0.00),
    )
    model.set_inlet(
        vessel_id='R-ICA_I',
        side='right',
        kind='pressure',
        function=make_pressure_inlet(85.0, 14.0, 4.0, phase_s=0.00),
    )
    model.set_inlet(
        vessel_id='BAS',
        side='left',
        kind='pressure',
        function=make_pressure_inlet(84.0, 11.0, 3.0, phase_s=0.04),
    )


def add_mid_probes(model, vessel_ids: tuple[str, ...]) -> None:
    for vessel_id in vessel_ids:
        if vessel_id not in model.config.vessels:
            continue
        length = model.config.vessel(vessel_id).length
        model.add_probe(vessel_id=vessel_id, position=0.5 * length, name='mid')


def configure_model(config_path: Path, *, probe_vessels: tuple[str, ...], method: str = 'CG'):
    model = hd.load_from_config(config_path)
    install_pressure_inlets(model)
    model.set_solver(
        method=method,
        h=0.0625,
        dt=1.0e-5,
        poly_order=1,
        dg_time_scheme='rk2',
        record_every=10,
    )
    add_mid_probes(model, probe_vessels)
    return model


def late_mask(times: np.ndarray, fraction: float = 0.25) -> np.ndarray:
    tail_start = times[-1] - fraction * (times[-1] - times[0])
    return times >= tail_start


def bed_late_means(results, bed_id: str) -> dict[str, float]:
    samples = results.capillary_bed_history(bed_id)
    times = np.array([s.time for s in samples])
    tail = late_mask(times)

    pcap_mmhg = np.array([s.pressure for s in samples]) / MMHG_TO_DYN_CM2
    inflow_ml_min = np.array([s.total_inflow for s in samples]) * 60.0
    venous_ml_min = np.array([s.venous_outflow for s in samples]) * 60.0
    perfusion = np.array([s.regional_perfusion for s in samples]) * 6000.0

    return {
        'Pcap [mmHg]': float(np.mean(pcap_mmhg[tail])),
        'Flow [mL/min]': float(np.mean(inflow_ml_min[tail])),
        'Venous out [mL/min]': float(np.mean(venous_ml_min[tail])),
        'Perfusion [mL/100g/min]': float(np.mean(perfusion[tail])),
        'Pcap min [mmHg]': float(np.min(pcap_mmhg[tail])),
        'Pcap max [mmHg]': float(np.max(pcap_mmhg[tail])),
    }


def summarize_beds(results, *, case: str) -> pd.DataFrame:
    rows = []
    for bed_id in results.capillary_bed_ids():
        row = {'case': case, 'bed': bed_id}
        row.update(bed_late_means(results, bed_id))
        rows.append(row)
    return pd.DataFrame(rows).sort_values(['case', 'bed'])


def probe_late_mean_flow(results, vessel_id: str, probe_name: str = 'mid') -> float | None:
    samples = results.history.probes.by_vessel_and_name(vessel_id, probe_name)
    if not samples:
        return None
    times = np.array([s.time for s in samples])
    tail = late_mask(times)
    q_ml_min = np.array([s.flow_rate for s in samples]) * 60.0
    return float(np.mean(q_ml_min[tail]))


def summarize_probe_flows(results, *, case: str, vessels: tuple[str, ...]) -> pd.DataFrame:
    rows = []
    for vessel_id in vessels:
        q = probe_late_mean_flow(results, vessel_id)
        if q is not None:
            rows.append({'case': case, 'vessel': vessel_id, 'mid flow [mL/min]': q})
    return pd.DataFrame(rows)


def plot_metric(summary: pd.DataFrame, *, x: str, y: str, hue: str | None = None, title: str = '') -> None:
    plt.figure(figsize=(8, 4.5))
    if hue is None:
        for bed, group in summary.groupby('bed'):
            plt.plot(group[x], group[y], marker='o', label=bed)
    else:
        for name, group in summary.groupby(hue):
            plt.plot(group[x], group[y], marker='o', label=str(name))
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
BASE_CONFIG_PATH = Path('experiments/configs/CoW_normal.json')
GENERATED_CONFIG_DIR = Path('experiments/configs/generated_notebooks/stent')
OUTPUT_ROOT = Path('experiments/outputs/notebooks/07_carotid_stent')
T_END = 2.0
RUN_SIMULATIONS = True

CASES = {
    'healthy': {'area_fraction': 1.00, 'beta_factor': 1.0},
    'pre_stent_85pct_area_stenosis': {'area_fraction': 0.15, 'beta_factor': 1.0},
    'post_stent_80pct_area_restored': {'area_fraction': 0.80, 'beta_factor': 5.0},
    'post_stent_full_area_restored': {'area_fraction': 1.00, 'beta_factor': 5.0},
}

PROBE_VESSELS = (
    'L-ICA_I', 'R-ICA_I', 'BAS',
    'ACA', 'L-PCommA', 'R-PCommA',
    'L-MCA', 'R-MCA', 'L-ACA_II', 'R-ACA_II',
)

In [ ]:
def make_stent_config(case_name: str, *, area_fraction: float, beta_factor: float) -> Path:
    config = read_config(BASE_CONFIG_PATH)
    config = copy.deepcopy(config)
    set_all_vessel_pressures(config, p0_mmhg=80.0, p_ext_mmhg=0.0)
    set_capillary_initial_pressure(config, 35.0)
    set_capillary_venous_pressure(config, 8.0)

    scale_area(config, 'L-ICA_I', area_fraction)
    scale_beta(config, 'L-ICA_I', beta_factor)

    path = GENERATED_CONFIG_DIR / f'CoW_{case_name}.json'
    return write_config(config, path)


def run_case(case_name: str, params: dict):
    config_path = make_stent_config(case_name, **params)
    model = configure_model(config_path, probe_vessels=PROBE_VESSELS)
    results = model.solve(t_end=T_END, show_progress=True, progress_description=case_name)

    output_dir = OUTPUT_ROOT / case_name
    results.save(output_dir)
    results.plot_capillary_beds(output_dir / 'plots', show=False)
    return results

In [ ]:
all_beds = []
all_flows = []

if RUN_SIMULATIONS:
    for case_name, params in CASES.items():
        results = run_case(case_name, params)
        beds = summarize_beds(results, case=case_name)
        beds['L-ICA area fraction'] = params['area_fraction']
        beds['L-ICA beta factor'] = params['beta_factor']
        all_beds.append(beds)

        flows = summarize_probe_flows(results, case=case_name, vessels=PROBE_VESSELS)
        flows['L-ICA area fraction'] = params['area_fraction']
        flows['L-ICA beta factor'] = params['beta_factor']
        all_flows.append(flows)

    bed_summary = pd.concat(all_beds, ignore_index=True)
    flow_summary = pd.concat(all_flows, ignore_index=True)
else:
    for case_name, params in CASES.items():
        make_stent_config(case_name, **params)
    bed_summary = pd.DataFrame()
    flow_summary = pd.DataFrame()

bed_summary

In [ ]:
if RUN_SIMULATIONS:
    left_anterior = bed_summary[bed_summary['bed'].isin(['L-MCA_bed', 'L-ACA_bed', 'R-MCA_bed', 'R-ACA_bed'])]
    display(left_anterior[['case', 'bed', 'Flow [mL/min]', 'Perfusion [mL/100g/min]', 'Pcap [mmHg]']])

    pivot = left_anterior.pivot_table(index='case', columns='bed', values='Perfusion [mL/100g/min]')
    display(pivot)
    pivot[['L-MCA_bed', 'L-ACA_bed', 'R-MCA_bed', 'R-ACA_bed']].plot(kind='bar', figsize=(9, 4), grid=True)
    plt.ylabel('Perfusion [mL/100g/min]')
    plt.title('Perfusion before and after L-ICA stent surrogate')
    plt.show()

In [ ]:
if RUN_SIMULATIONS:
    collateral = flow_summary[flow_summary['vessel'].isin(['L-ICA_I', 'R-ICA_I', 'ACA', 'L-PCommA'])]
    display(collateral)

## Expected result

The severe pre-stent case should reduce `L-MCA_bed` and `L-ACA_bed` perfusion and recruit collateral flow. The post-stent cases should recover left anterior perfusion and reduce the need for collateral compensation.

This is still a **structural surrogate**. A more realistic stent model would split `L-ICA_I` into pre-stent, stent, and post-stent segments and add a localized nonlinear pressure loss before treatment.